# Autoencoder - Compression & Reconstruction - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## الهدف التعليمي
Autoencoder يضغّط الصورة (784 بكسل) إلى **32 بعد** (latent) ثم يعيد بناءها.
تعلم **غير مُوجَّه** (unsupervised): الهدف = إعادة بناء X نفسه.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات
3. تجهيز الصور
4. تقسيم البيانات
5. بناء Autoencoder
6. compile + fit
7. إعادة البناء
8. original vs reconstructed
9. فضاء latent


## الخطوة 1: استيراد المكتبات


In [ ]:
# الخطوة 1) استيراد المكتبات
# pip install tensorflow -q
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense


## الخطوة 2: قراءة البيانات


In [ ]:
# Step 2) قراءة البيانات / Load dataset
import os
import urllib.request

filename = 'mnist_sample.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/15-%20Deep%20Learning/7-%20Autoencoder/mnist_sample.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset.head()


## الخطوة 3: تجهيز الصور

نفرد الصورة لمتجه 784 ونقسم على 255 للـ normalization.


In [ ]:
# الخطوة 3) تجهيز الصور
pixel_cols = [c for c in dataset.columns if c.startswith('pixel_')]
X = dataset[pixel_cols].values.astype('float32') / 255.0
print('X shape:', X.shape)


## الخطوة 4: تقسيم البيانات

لا نحتاج y — التعلم غير مُوجَّه.


In [ ]:
# الخطوة 4) تقسيم البيانات
X_train, X_test = train_test_split(X, test_size=0.2, random_state=0)
print('Train:', X_train.shape, '| Test:', X_test.shape)


## الخطوة 5: بناء Autoencoder

**Encoder:** 784 → 128 → 64 → **32 (bottleneck/latent)**
**Decoder:** 32 → 64 → 128 → 784

Bottleneck يُجبر النموذج على تعلم تمثيل مضغوط.


In [ ]:
# الخطوة 5) بناء Autoencoder
input_dim = X.shape[1]
latent_dim = 32

inputs = Input(shape=(input_dim,))
encoded = Dense(128, activation='relu')(inputs)
encoded = Dense(64, activation='relu')(encoded)
latent = Dense(latent_dim, activation='relu', name='latent')(encoded)
decoded = Dense(64, activation='relu')(latent)
decoded = Dense(128, activation='relu')(decoded)
outputs = Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = Model(inputs, outputs)
encoder = Model(inputs, latent)
autoencoder.summary()


## الخطوة 6: compile + fit

- **loss=mse**: Mean Squared Error بين الأصل والمعاد بناء
- **X_train, X_train**: الهدف = المدخلات (reconstruction)


In [ ]:
# الخطوة 6) compile + fit
autoencoder.compile(optimizer='adam', loss='mse')
history = autoencoder.fit(
    X_train, X_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1
)


## الخطوة 7: إعادة البناء

MSE منخفض = إعادة بناء جيدة.


In [ ]:
# الخطوة 7) إعادة البناء
reconstructed = autoencoder.predict(X_test, verbose=0)
test_mse = np.mean((X_test - reconstructed) ** 2)
print(f'Test reconstruction MSE: {test_mse:.6f}')


## الخطوة 8: Original vs Reconstructed

قارن الصف العلوي (أصل) بالسفلي (معاد بناء).
التفاصيل الدقيقة قد تُفقد بسبب الضغط 784→32.


In [ ]:
# الخطوة 8) original vs reconstructed
n = 5
fig, axes = plt.subplots(2, n, figsize=(12, 4))
for i in range(n):
    axes[0, i].imshow(X_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray')
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')
plt.suptitle('Autoencoder: Original vs Reconstructed')
plt.tight_layout()
plt.show()


## الخطوة 9: فضاء Latent

نرسم أول بعدين من latent codes (32 بعد — نعرض 2 للتصور).
في التطبيق: clustering، anomaly detection، generation.


In [ ]:
# الخطوة 9) فضاء latent
latent_codes = encoder.predict(X_test[:100], verbose=0)
plt.figure(figsize=(5, 4))
plt.scatter(latent_codes[:, 0], latent_codes[:, 1], alpha=0.7, s=20)
plt.xlabel('Latent dim 1')
plt.ylabel('Latent dim 2')
plt.title('Latent Space (first 2 dimensions)')
plt.show()
print(f'Bottleneck compresses {input_dim} pixels -> {latent_dim} dimensions.')
